In [4]:
import os
import sys

# Append the src directory to the python path so we can import our modules
sys.path.append(os.path.abspath(os.path.join('..')))

import yaml
from src.mifid_engine import load_questionnaire, calculate_mifid_profile

In [5]:
# ==============================================================================
# 1. TESTING INTERVIEW LOADING (YAML CONFIGURATION)
# ==============================================================================
print("====== STEP 1: LOADING QUESTIONNAIRE ======")
yaml_path = "../config/questionnaire.yaml"  # Adjust path if running from root or src

try:
    raw_config = load_questionnaire(yaml_path)
    print(f"Successfully loaded: {raw_config['metadata']['questionnaire_name']}")
    print(f"Version: {raw_config['metadata']['version']}\n")
    
    # Let's inspect how many questions were loaded per pillar
    print("Questions loaded from YAML:")
    for q in raw_config['questions']:
        print(f" - [{q['pillar'].upper()}] ID: {q['id']} -> {q['text'][:50]}...")
        
except Exception as e:
    print(f"Error during loading: {e}")

print("\n" + "="*50 + "\n")

====== STEP 1: LOADING QUESTIONNAIRE ======
Successfully loaded: MiFID Risk Profiler - RiskAlign
Version: 1.1

Questions loaded from YAML:
 - [KNOWLEDGE_EXPERIENCE] ID: q_exp_education -> Qual è il tuo livello di background formativo o pr...
 - [KNOWLEDGE_EXPERIENCE] ID: q_exp_frequency -> Negli ultimi 2 anni, con quale frequenza hai effet...
 - [KNOWLEDGE_EXPERIENCE] ID: q_exp_derivatives -> Qual è il tuo livello di familiarità con gli strum...
 - [FINANCIAL_SITUATION] ID: q_fin_income_stability -> Qual è la natura e la stabilità delle tue fonti di...
 - [FINANCIAL_SITUATION] ID: q_fin_loss_capacity -> In caso di perdite rilevanti sul tuo portafoglio d...
 - [FINANCIAL_SITUATION] ID: q_fin_wealth_pct -> Qual è la quota del tuo patrimonio totale liquidab...
 - [INVESTMENT_OBJECTIVES] ID: q_obj_horizon -> Per quanto tempo intendi mantenere investito quest...
 - [INVESTMENT_OBJECTIVES] ID: q_obj_target -> Qual è l'obiettivo primario che ti prefiggi con qu...
 - [INVESTMENT_OBJECTIVES] ID

In [12]:
# Simulation: Balanced Client (Moderate experience, solid finances, medium-term objectives)
balanced_client_answers = {
    "q_exp_education": "a2",        # General knowledge (3)
    "q_exp_frequency": "a3",        # Moderate trading (4)
    "q_exp_derivatives": "a2",      # Theoretical knowledge of derivatives (2)
    
    "q_fin_income_stability": "a1",  # Stable income (5)
    "q_fin_loss_capacity": "a1",     # Can bear temporary losses (3)
    "q_fin_wealth_pct": "a1",        # Investing 30%-70% of wealth (3)
    
    "q_obj_horizon": "a2",          # Medium term (3)
    "q_obj_target": "a2",           # Moderate growth (3)
    "q_obj_reaction": "a2"          # Monitors with concern but waits (3)
}

In [13]:
mifid_results = calculate_mifid_profile(balanced_client_answers, yaml_path)

print("Engine Output for Balanced Client:")
print(f" -> Pillar Averages: {mifid_results['pillar_averages']}")
print(f" -> Raw Weighted Score: {mifid_results['raw_weighted_score']}")
print(f" -> Final SRI Profile: {mifid_results['user_sri_profile']} (out of 7)")
print(f" -> Capping Triggered?: {mifid_results['capping_triggered']}")

Engine Output for Balanced Client:
 -> Pillar Averages: {'knowledge_experience': 3.0, 'financial_situation': 1.0, 'investment_objectives': 3.0}
 -> Raw Weighted Score: 2.2
 -> Final SRI Profile: 2 (out of 7)
 -> Capping Triggered?: True


In [15]:
def validate_portfolio(portfolio: dict) -> bool:
    """
    Validates the portfolio structure and checks if total weights equal 1.0 (100%).
    """
    print(f"Reviewing submitted portfolio containing {len(portfolio)} assets...")
    
    # Check for empty portfolio
    if not portfolio:
        print("Error: Portfolio is empty.")
        return False
        
    total_weight = sum(portfolio.values())
    print(f" -> Total calculated weight: {total_weight:.2f} ({total_weight * 100:.1f}%)")
    
    # In finance data science, we allow a tiny floating-point margin (tolerance)
    tolerance = 1e-5
    if abs(total_weight - 1.0) > tolerance:
        print("Error: Total portfolio weights must equal 1.0 (100%).")
        return False
        
    # Check for negative weights (unless short-selling is allowed, which we don't)
    for asset, weight in portfolio.items():
        if weight < 0:
            print(f"Error: Negative weight detected for asset {asset}.")
            return False
            
    print("Success: Portfolio structure is valid and ready for quantitative analysis.")
    return True



In [19]:
# ==============================================================================
# 3. TESTING CLIENT PORTFOLIO LOADING & VALIDATION
# ==============================================================================
print("====== STEP 3: LOADING & VALIDATING CLIENT PORTFOLIO ======")

# Simulation: A client submits a portfolio dictionary containing Tickers and Weights
# The weights represent the percentage allocation (0.0 to 1.0)
client_portfolio = {
    "AAPL": 0.40000001,   # 40% Apple (US Equity)
    "MSFT": 0.30,   # 30% Microsoft (US Equity)
    "IE00B4L5Y983": 0.20, # 20% MSCI World ETF (Global Equity)
    "BTC-USD": 0.10  # 10% Bitcoin (Crypto)
}

====== STEP 3: LOADING & VALIDATING CLIENT PORTFOLIO ======


In [20]:
is_valid = validate_portfolio(client_portfolio)

Reviewing submitted portfolio containing 4 assets...
 -> Total calculated weight: 1.00 (100.0%)
Success: Portfolio structure is valid and ready for quantitative analysis.
